# Geospatial Features Extraction for Nigerian Cities

**Notebook 01a: JRC Water Bodies + OSM Roads & Places**

This notebook extracts geospatial features for 58 city AOIs using:
- **JRC Global Surface Water** (via Google Earth Engine)  Water bodies with â‰¥50% occurrence
- **OpenStreetMap** (via Overpass API)  Roads and place names

---

## Table of Contents

1. [Load City AOIs](#1-load-city-aois)  Load AOI boundaries and check tracking status
2. [Extract Features for Each City](#2-extract-features-for-each-city)  Download water (JRC), roads (OSM), and places (OSM)
3. [Visualize Geospatial Features](#3-visualize-geospatial-features)  Create individual city maps
4. [Publication-Ready Study Area Map](#4-publication-ready-study-area-map)  Multi-panel figure for manuscripts
5. [Nigeria Cities & Landscape Overview Map](#5-nigeria-cities--landscape-overview-map) €” Landscape classification map + Cluster25 table
6. [Summary](#summary) €” Workflow recap and next steps

---

## Data Sources

| Feature | Source | Resolution | Method |
|---------|--------|------------|--------|
| **Water Bodies** | JRC Global Surface Water | 30m | Google Earth Engine (â‰¥50% occurrence) |
| **Roads** | OpenStreetMap | Vector | Overpass API via `osmnx` |
| **Places** | OpenStreetMap | Vector | Overpass API via `osmnx` |
| **Landscape** | Cluster25 Classification | Vector | GeoJSON polygons with climate, geology, elevation classes |
| **Population** | UN WUP 2025 | City-level | CSV with annual estimates |

> **Note**: JRC water bodies are used instead of OSM water features because JRC provides consistent, satellite-derived water extent data that captures seasonal water bodies and lagoons more reliably.

---
## Skip Logic

- **Area tracking**: Features only re-extracted if AOI area changes (tracked via `aoi_features_tracking.json`)
- **File checking**: Individual feature files verified before extraction
- **Output checking**: Publication figures and landscape maps skipped if SVG + SVG already exist

---

## Prerequisites

This notebook requires:
- **Notebook 01** completed first (AOI boundaries must exist)
- **Google Earth Engine** authentication for JRC water extraction
- **osmnx** package for OSM queries (`pip install osmnx`)

---
## 1. Load City AOIs

Load all city AOI boundaries from the `01_aoi_delineation` output folder and check which cities need feature updates based on tracking data.

In [ ]:
# ====================================================================
# IMPORT LIBRARIES AND CONFIGURE
# ====================================================================

import ee
import glob
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import sys
import time
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / 'src'))

# Import configuration and modules
import config
from src.gee_utils import initialize_gee
from src.data_io import (
    load_aoi_tracking, save_aoi_tracking, update_aoi_tracking,
    check_features_need_update, check_osm_feature_exists,
    load_city_osm_features, load_city_polygon_features,
    get_osm_extraction_status, get_osm_features_summary, 
    filter_places_for_display
)
from src.visualization import create_geospatial_features_map

print("="*80)
print("GEOSPATIAL FEATURES EXTRACTION")
print("="*80)
print(f"Project root: {project_root}")

# Initialize Google Earth Engine
service_key = config.GEE_CONFIG['service_json']
project_id = config.GEE_CONFIG['project_id']

GEE_AVAILABLE = initialize_gee(service_key, project_id)
if not GEE_AVAILABLE:
    print("  Water features will be skipped")

# Import osmnx for OSM queries via Overpass API
OSMNX_AVAILABLE = False
try:
    import osmnx as ox
    ox.settings.timeout = 180  # 3 min timeout
    ox.settings.memory = None
    ox.settings.log_console = False
    print("osmnx available for Overpass API queries (10-30s per city)")
    OSMNX_AVAILABLE = True
except ImportError:
    print("osmnx not available - install with: pip install osmnx")

print(f"GeoPandas version: {gpd.__version__}")
print("="*80)


In [ ]:
# ====================================================================
# LOAD AOI BOUNDARIES & CHECK TRACKING STATUS
# ====================================================================

import json

VERBOSE_LIMIT = 5  # Only print details for first N cities

aoi_base_dir = project_root / 'data' / 'outputs' / '01_aoi_delineation'

print(f"\nLoading AOI boundaries from: {aoi_base_dir}")
print("-"*80)

# Load the AOI manifest from Notebook 01 (58 deduplicated AOIs)
manifest_path = aoi_base_dir / 'aoi_manifest.json'
if manifest_path.exists():
    with open(manifest_path) as f:
        manifest = json.load(f)
    valid_cities = set(manifest['aoi_cities'])
    absorbed = manifest.get('absorbed_cities', {})
    print(f" Loaded AOI manifest: {manifest['n_aois']} deduplicated AOIs")
    if absorbed:
        print(f"   ({len(absorbed)} cities absorbed into parent AOIs)")
else:
    # Fallback: use config selection (no dedup info available)
    valid_cities = set(config.get_cities_to_process())
    absorbed = {}
    print(f"âš ï¸  No AOI manifest found  using config selection ({len(valid_cities)} cities)")
    print(f"   Run Notebook 01 first to generate the manifest.")

aoi_dict = {}
# AOI files are in City_*/Features/ subdirectory
aoi_pattern = str(aoi_base_dir / 'City_*' / 'Features' / '*_AOI_Boundary.geojson')
aoi_files = glob.glob(aoi_pattern)

if not aoi_files:
    raise FileNotFoundError(f"No AOI boundary files found in {aoi_base_dir}/City_*/Features/")

# Filter to only cities in the manifest (58 AOIs, not the raw 67)
aoi_files_filtered = []
skipped_cities = []
for f in sorted(aoi_files):
    city_name = Path(f).stem.replace('_AOI_Boundary', '')
    if city_name in valid_cities:
        aoi_files_filtered.append(f)
    else:
        skipped_cities.append(city_name)

if skipped_cities:
    print(f"Skipped {len(skipped_cities)} non-AOI city folders (absorbed or stale)")

aoi_files = aoi_files_filtered

# Load current tracking data
tracking_data = load_aoi_tracking(aoi_base_dir)
print(f"\nTracking file: {len(tracking_data)} cities previously tracked")

update_count = 0
skip_count = 0

for idx, file in enumerate(sorted(aoi_files)):
    city_name = Path(file).stem.replace('_AOI_Boundary', '')
    gdf = gpd.read_file(file)

    # Ensure WGS84 for OSM queries
    if gdf.crs != 'EPSG:4326':
        gdf = gdf.to_crs('EPSG:4326')

    aoi_dict[city_name] = gdf
    bounds = gdf.total_bounds

    # Check if features need update (uses Main_Polygon area for comparison)
    needs_update, reason, current_area, tracked_area = check_features_need_update(city_name, aoi_base_dir)

    if needs_update:
        update_indicator = "ðŸ”„ UPDATE NEEDED"
        update_count += 1
    else:
        update_indicator = "Up to date"
        skip_count += 1

    # Display Main_Polygon area (only first VERBOSE_LIMIT cities)
    if idx < VERBOSE_LIMIT:
        print(f"{update_indicator} {city_name:15s} | Main Polygon: {current_area:7.2f} km² | {reason}")

if len(aoi_files) > VERBOSE_LIMIT:
    print(f"  ... and {len(aoi_files) - VERBOSE_LIMIT} more cities loaded (details suppressed)")

cities_to_process = sorted(aoi_dict.keys())

print("-"*80)
print(f"Loaded {len(aoi_dict)} city AOIs (from {manifest['n_aois']} in AOI manifest)")
if skip_count == len(cities_to_process):
    print(f" All {skip_count} cities up-to-date - feature extraction will be SKIPPED")
elif update_count > 0:
    print(f"{update_count} cities need updates, {skip_count} will be skipped")
print("="*80)

In [ ]:
# ====================================================================
# PRE-EXTRACTION STATUS CHECK
# ====================================================================

import warnings
import logging

# Suppress OGR/fiona warnings about unsupported field types
warnings.filterwarnings('ignore', message='.*Skipping field.*')
warnings.filterwarnings('ignore', message='.*unsupported OGR type.*')
logging.getLogger('fiona').setLevel(logging.ERROR)

print(f"\n{'='*80}")
print("FEATURE EXTRACTION STATUS CHECK")
print(f"{'='*80}")

# Reuse update_count / skip_count from cell above (already computed)
if skip_count == len(cities_to_process):
    # All cities up-to-date  no need to re-read every feature file
    print(f"\nâœ“ All {skip_count} cities up-to-date  skipping detailed status check.")
    print(f"  (Feature files already verified during AOI loading above.)")
else:
    # Only show details for cities that actually need updates
    shown = 0
    for city in cities_to_process:
        status = get_osm_extraction_status(city, aoi_base_dir)
        if not status['needs_update']:
            continue
        shown += 1

        if shown <= VERBOSE_LIMIT:
            print(f"\nðŸ”„ {city}:")
            print(f"  AOI Area: {status['current_area_km2']:.2f} km²")
            print(f"  Reason: {status['update_reason']}")
            for feat_type, feat_info in status['features'].items():
                if feat_info['exists']:
                    print(f"  {feat_type}: {feat_info['count']} features")
                else:
                    print(f"   {feat_type}: Not extracted")

    if shown > VERBOSE_LIMIT:
        print(f"\n  ... and {shown - VERBOSE_LIMIT} more cities need updates (details suppressed)")

    print(f"\n  Summary: {update_count} need updates, {skip_count} up-to-date")

print(f"\nNOTE: Only features needing update will be extracted")
print(f"{'='*80}")

---
## 2. Extract Features for Each City

For each city AOI, extract three feature types:

| Feature | Source | Method | Threshold |
|---------|--------|--------|-----------|
| **Water Bodies** | JRC Global Surface Water | Google Earth Engine | â‰¥50% occurrence (captures lagoons) |
| **Roads** | OpenStreetMap | Overpass API via `osmnx` | All highway types |
| **Places** | OpenStreetMap | Overpass API via `osmnx` | All place types |

> **Skip Logic**: If a city's AOI area hasn't changed since the last extraction (tracked in `aoi_features_tracking.json`), that city will be skipped to save time.

In [ ]:
# ====================================================================
# EXTRACT FEATURES FOR EACH CITY (WITH SKIP LOGIC)
# ====================================================================

for i, city in enumerate(cities_to_process, 1):
    print(f"\n{'='*80}")
    print(f"PROCESSING {city.upper()} ({i}/{len(cities_to_process)})")
    print(f"{'='*80}")
    
    gdf = aoi_dict[city]
    
    # Check if this city needs any updates
    needs_update, reason, current_area, tracked_area = check_features_need_update(city, aoi_base_dir)
    
    # Create output directory
    city_osm_dir = aoi_base_dir / f'City_{city}' / 'Features' / 'OSM'
    city_osm_dir.mkdir(parents=True, exist_ok=True)
    
    # Get the AOI polygon for queries
    aoi_polygon = gdf.to_crs('EPSG:4326').union_all()
    
    # Track if any features were extracted
    features_extracted = False
    
    # ====================================================================
    # 1. JRC SURFACE WATER (FROM GEE)
    # ====================================================================
    water_exists, water_file, water_count = check_osm_feature_exists(city, 'water', aoi_base_dir)
    
    if water_exists and not needs_update:
        print(f"\n1. Water bodies: SKIP ({water_count} features exist, AOI unchanged)")
    elif not GEE_AVAILABLE:
        print(f"\n1. Water bodies:  GEE not available")
    else:
        print(f"\n1. Downloading JRC surface water for {city}...")
        try:
            water_file_path = city_osm_dir / f'{city}_water_bodies.geojson'
            
            # Convert AOI to Earth Engine geometry
            aoi_geojson = gdf.to_crs('EPSG:4326').__geo_interface__
            ee_geometry = ee.Geometry(aoi_geojson['features'][0]['geometry'])
            
            # Load JRC Global Surface Water
            jrc_water = ee.Image('JRC/GSW1_4/GlobalSurfaceWater')
            water_occurrence = jrc_water.select('occurrence')
            water_mask = water_occurrence.gte(50)
            water_masked = water_mask.updateMask(water_mask.eq(1))
            
            # Convert to vectors
            water_vectors = water_masked.reduceToVectors(
                geometry=ee_geometry,
                scale=30,
                geometryType='polygon',
                eightConnected=False,
                maxPixels=1e9
            )
            
            water_features = water_vectors.getInfo()
            
            if water_features['features']:
                water_gdf = gpd.GeoDataFrame.from_features(water_features['features'], crs='EPSG:4326')
                water_clipped = gpd.clip(water_gdf, gdf.to_crs('EPSG:4326'))
                
                if not water_clipped.empty:
                    water_clipped['area_km2'] = water_clipped.to_crs('EPSG:32632').area / 1e6
                    water_clipped = water_clipped[water_clipped['area_km2'] >= 0.001]
                    
                    if not water_clipped.empty:
                        water_clipped.to_file(water_file_path, driver='GeoJSON')
                        total_area = water_clipped['area_km2'].sum()
                        print(f"   Saved {len(water_clipped)} water bodies ({total_area:.2f} km²)")
                        features_extracted = True
                    else:
                        print(f"   - No significant water bodies after filtering")
                else:
                    print(f"   - No water bodies within AOI")
            else:
                print(f"   - No water detected by JRC")
                
        except Exception as e:
            print(f"    Error: {e}")
    
    # ====================================================================
    # 2. OSM ROADS (VIA OVERPASS API)
    # ====================================================================
    roads_exists, roads_file, roads_count = check_osm_feature_exists(city, 'roads', aoi_base_dir)
    
    if roads_exists and not needs_update:
        print(f"\n2. Roads: SKIP ({roads_count} features exist, AOI unchanged)")
    elif not OSMNX_AVAILABLE:
        print(f"\n2. Roads:  osmnx not available")
    else:
        print(f"\n2. Extracting OSM roads for {city}...")
        try:
            roads_file_path = city_osm_dir / f'{city}_roads.geojson'
            start_time = time.time()
            
            G = ox.graph_from_polygon(aoi_polygon, network_type='drive', simplify=True)
            roads = ox.graph_to_gdfs(G, nodes=False)
            elapsed = time.time() - start_time
            
            if not roads.empty:
                roads = roads.to_crs('EPSG:4326')
                roads.to_file(roads_file_path, driver='GeoJSON')
                print(f"   Saved {len(roads)} road segments in {elapsed:.1f}s")
                features_extracted = True
                
                if 'highway' in roads.columns:
                    highway_flat = roads['highway'].apply(
                        lambda x: x[0] if isinstance(x, (list, tuple)) else x
                    )
                    road_types = highway_flat.value_counts().head(5)
                    print(f"   Top types: {dict(road_types)}")
            else:
                print(f"   - No roads found")
                
        except Exception as e:
            elapsed = time.time() - start_time
            print(f"    Error after {elapsed:.1f}s: {e}")
    
    # ====================================================================
    # 3. OSM PLACES (VIA OVERPASS API)
    # ====================================================================
    places_exists, places_file, places_count = check_osm_feature_exists(city, 'places', aoi_base_dir)
    
    if places_exists and not needs_update:
        print(f"\n3. Places: SKIP ({places_count} features exist, AOI unchanged)")
    elif not OSMNX_AVAILABLE:
        print(f"\n3. Places:  osmnx not available")
    else:
        print(f"\n3. Extracting OSM places for {city}...")
        try:
            places_file_path = city_osm_dir / f'{city}_places.geojson'
            start_time = time.time()
            
            place_tags = {'place': True}
            places = ox.features_from_polygon(aoi_polygon, tags=place_tags)
            elapsed = time.time() - start_time
            
            if not places.empty:
                places = places.to_crs('EPSG:4326')
                places.to_file(places_file_path, driver='GeoJSON')
                print(f"   Saved {len(places)} places in {elapsed:.1f}s")
                features_extracted = True
                
                if 'place' in places.columns:
                    place_types = places['place'].value_counts().head(5)
                    print(f"   Top types: {dict(place_types)}")
                
                if 'name' in places.columns:
                    named_places = places[places['name'].notna()]['name'].head(5)
                    if not named_places.empty:
                        print(f"   Sample names: {', '.join(named_places.tolist())}")
            else:
                print(f"   - No places found")
                
        except Exception as e:
            elapsed = time.time() - start_time
            if "EmptyOverpassResponse" in str(type(e).__name__) or "no data" in str(e).lower():
                print(f"   - No places found")
            else:
                print(f"    Error after {elapsed:.1f}s: {e}")
    
    # ====================================================================
    # UPDATE TRACKING IF FEATURES WERE EXTRACTED
    # ====================================================================
    if features_extracted or needs_update:
        update_aoi_tracking(city, current_area, aoi_base_dir)
        print(f"\n   Tracking updated for {city} (area: {current_area:.2f} km²)")

print(f"\n{'='*80}")
print("FEATURE EXTRACTION COMPLETE")
print(f"{'='*80}")

---
## 3. Visualize Geospatial Features

Create maps showing extracted features for each city:

| Feature | Styling | Filter |
|---------|---------|--------|
| **Water** | Blue fill (`steelblue`) with dark edge | JRC â‰¥50% occurrence |
| **Roads** | Thin grey lines | All highway types |
| **Places** | Dark red markers with labels | Towns and suburbs only |

All visualization outputs are saved in both **SVG** and **SVG** formats to `City_[Name]/Figures/`.

In [ ]:
# ====================================================================
# SUMMARY OF DOWNLOADED FEATURES
# ====================================================================

# Suppress OGR 'Skipping field' warnings from fiona
import logging
logging.getLogger('fiona').setLevel(logging.ERROR)
logging.getLogger('fiona._err').setLevel(logging.ERROR)

print(f"\n{'='*80}")
print("GEOSPATIAL FEATURES SUMMARY")
print(f"{'='*80}")

# Get summary table using modular function
summary_df = get_osm_features_summary(cities_to_process, aoi_base_dir)

# Display formatted summary (first VERBOSE_LIMIT cities only)
for row_idx, (_, row) in enumerate(summary_df.iterrows()):
    if row_idx >= VERBOSE_LIMIT:
        break
    city = row['city']
    print(f"\n{city}:")
    print("-"*60)

    total_size = 0
    for feat in ['water', 'roads', 'places']:
        count = row[f'{feat}_count']
        size = row[f'{feat}_size_mb']
        total_size += size

        if count > 0:
            print(f"  \u2713 {feat:10s}: {count:5d} features | {size:6.2f} MB")
        else:
            print(f"  \u2717 {feat:10s}: Not available")

    print(f"  {'\u2500'*50}")
    print(f"  Total: {total_size:.2f} MB")

if len(summary_df) > VERBOSE_LIMIT:
    print(f"\n  ... and {len(summary_df) - VERBOSE_LIMIT} more cities (details suppressed)")

# Aggregate totals
total_water = summary_df['water_count'].sum()
total_roads = summary_df['roads_count'].sum()
total_places = summary_df['places_count'].sum()
total_all_size = summary_df[['water_size_mb', 'roads_size_mb', 'places_size_mb']].sum().sum()
print(f"\nAggregate totals ({len(summary_df)} cities): water={total_water}, roads={total_roads}, places={total_places} | {total_all_size:.1f} MB")

# Store feature paths for visualization
geo_features = {}
for city in cities_to_process:
    geo_features[city] = load_city_osm_features(city, aoi_base_dir)

# Load polygon features for all cities
polygon_features = {}
for city in cities_to_process:
    polygon_features[city] = load_city_polygon_features(city, aoi_base_dir)

print(f"\n{'='*80}")
print(f"Output location: {aoi_base_dir}/City_[Name]/Features/OSM/")
print(f"Tracking file: {aoi_base_dir}/aoi_features_tracking.json")
print(f"{'='*80}")


In [ ]:
# ====================================================================
# VISUALIZE GEOSPATIAL FEATURES FOR EACH CITY
# ====================================================================

import logging
logging.getLogger('fiona').setLevel(logging.ERROR)
logging.getLogger('fiona._err').setLevel(logging.ERROR)

print(f"\n{'='*80}")
print("GEOSPATIAL FEATURES VISUALIZATION")
print(f"{'='*80}\n")

skipped = 0
generated = 0
for idx, city in enumerate(cities_to_process):
    verbose = (idx < VERBOSE_LIMIT)

    # Get AOI boundary
    aoi_gdf = aoi_dict[city]

    # Get features (already loaded)
    features = geo_features[city]
    polygons = polygon_features[city]

    # Filter places to towns and suburbs only
    places_filtered = filter_places_for_display(
        features.get('places'),
        place_types=['town', 'suburb']
    )

    # Create output path
    fig_dir = aoi_base_dir / f'City_{city}' / 'Figures'
    fig_dir.mkdir(parents=True, exist_ok=True)
    fig_path = fig_dir / f'{city}_geospatial_features_map.svg'
    #pdf_path = fig_dir / f'{city}_geospatial_features_map.svg'

    # Skip if both SVG and SVG already exist
    if fig_path.exists() and pdf_path.exists():
        skipped += 1
        continue

    generated += 1
    if verbose:
        print(f"Visualizing {city}...")

    # Create visualization using modular function with polygon layers
    fig = create_geospatial_features_map(
        city_name=city,
        aoi_gdf=aoi_gdf,
        main_polygon_gdf=polygons.get('main_polygon'),
        adjacent_polygons_gdf=polygons.get('adjacent_polygons'),
        water_gdf=features.get('water'),
        roads_gdf=features.get('roads'),
        places_gdf=places_filtered,
        place_types_filter=['town', 'suburb'],
        label_buffer_km=1.4,
        figsize=(12, 10),
        save_path=str(fig_path),
        dpi=300
    )

    if verbose:
        plt.show()
    plt.close()

if skipped > 0:
    print(f"\n  \u2713 {skipped} cities already had maps on disk \u2014 skipped.")
if generated > 0:
    shown = min(generated, VERBOSE_LIMIT)
    print(f"  \u2713 {generated} new maps generated ({shown} shown inline).")
else:
    print(f"\n  \u2713 All {len(cities_to_process)} city maps already exist \u2014 nothing to regenerate.")

print(f"\n{'='*80}")
print("VISUALIZATION COMPLETE")
print(f"{'='*80}")
print("\n\u2713 All maps saved to SVG and SVG formats")
print("\u2713 JRC water bodies will be automatically masked during SEVIRI processing")
print("\u2713 OSM roads and places available for overlay visualization")


---
## 4. Publication-Ready Study Area Map

Map output varies by city selection mode (configured in `config.py`):

| Mode | Map Output |
|------|------------|
| **TEST** (5 cities) | Combined multi-panel figure: Nigeria inset + all cities in one A4 landscape |
| **ALL / ABOVE_1M / SPECIFIC** | Individual maps: one Nigeria overview + separate A4 landscape per city (saved in each city's `Figures/` folder) |

| Element | Description |
|---------|-------------|
| **Nigeria inset** | Country outline with AOI locations and city labels |
| **City panels** | JRC water, OSM roads, OSM places with scale bars |
| **Layout** | A4 landscape format (11.69" Ã— 7.5") |
| **Cartographic elements** | Scale bars inside map frames, north arrow, panel labels (A, B, C...) |

Output locations:
- **TEST**: `Study_Area_Map_Publication.svg` (single combined figure)
- **Other modes**: `City_[Name]/Figures/[Name]_Study_Area_Map.svg` (one per city) + `Nigeria_Cities_Overview.svg`

In [ ]:
# ====================================================================
# PUBLICATION-READY STUDY AREA MAP
# Conditional: Combined (TEST) vs Individual (ALL/ABOVE_1M/SPECIFIC)
# ====================================================================

import importlib
import logging
import src.visualization
importlib.reload(src.visualization)
from src.visualization import (
    create_publication_study_area_map,
    create_individual_city_study_area_map,
    create_nigeria_overview_map,
    create_nigeria_cities_landscape_map,
    create_cluster25_classification_table
)

# Suppress OGR warnings
logging.getLogger('fiona').setLevel(logging.ERROR)
logging.getLogger('fiona._err').setLevel(logging.ERROR)

print(f"\n{'='*80}")
print("CREATING PUBLICATION-READY STUDY AREA MAP(S)")
print(f"{'='*80}")

# Define paths
nigeria_path = project_root / 'data' / 'inputs' / 'geospatial_features' / 'boundary_Nigeria.geojson'
is_test_mode = config.CITY_SELECTION_MODE == "TEST"

print(f"City selection mode: {config.CITY_SELECTION_MODE}")
print(f"Number of cities: {len(cities_to_process)}")

if is_test_mode:
    # ================================================================
    # TEST MODE: Combined multi-panel figure
    # ================================================================
    print(f"\u2192 Creating COMBINED multi-panel map (TEST mode)\n")

    save_path = aoi_base_dir / 'Study_Area_Map_Publication'
    png_path = save_path.with_suffix('.svg')
   #pdf_path = save_path.with_suffix('.svg')

    if png_path.exists() and pdf_path.exists():
        print(f"  \u2713 Combined map already exists \u2014 skipped.")
    else:
        fig = create_publication_study_area_map(
            aoi_dict=aoi_dict,
            geo_features=geo_features,
            polygon_features=polygon_features,
            nigeria_path=nigeria_path,
            save_path=save_path,
            title='Study Area Maps: Nigerian Cities with Geospatial Features',
            figsize=(11.69, 7.5),
            dpi=300
        )
        plt.show()
        plt.close()

else:
    # ================================================================
    # NON-TEST MODE: Individual maps per city
    # ================================================================
    print(f"\u2192 Creating INDIVIDUAL maps per city ({config.CITY_SELECTION_MODE} mode)\n")

    # 1. Nigeria overview showing all selected cities
    overview_path = aoi_base_dir / 'Nigeria_Cities_Overview'
    overview_png = overview_path.with_suffix('.svg')
    overview_pdf = overview_path.with_suffix('.svg')

    if overview_png.exists() and overview_pdf.exists():
        print(f"  \u2713 Nigeria overview map already exists \u2014 skipped.")
    else:
        print(f"{'\u2500'*60}")
        print("Creating Nigeria overview map...")
        fig_overview = create_nigeria_overview_map(
            aoi_dict=aoi_dict,
            nigeria_path=nigeria_path,
            save_path=overview_path,
            title=f'Study Cities: Nigeria ({len(cities_to_process)} cities)',
            figsize=(11.69, 7.5),
            dpi=300
        )
        plt.show()
        plt.close()

    # 2. Individual city maps
    skipped = 0
    generated = 0
    for i, city in enumerate(sorted(cities_to_process), 1):
        verbose = (i <= VERBOSE_LIMIT)

        fig_dir = aoi_base_dir / f'City_{city}' / 'Figures'
        fig_dir.mkdir(parents=True, exist_ok=True)
        city_save_path = fig_dir / f'{city}_Study_Area_Map'
        city_png = city_save_path.with_suffix('.svg')
        city_pdf = city_save_path.with_suffix('.svg')

        if city_png.exists() and city_pdf.exists():
            skipped += 1
            continue

        generated += 1
        if verbose:
            print(f"\n{'\u2500'*60}")
            print(f"Creating map {i}/{len(cities_to_process)}: {city}")

        fig_city = create_individual_city_study_area_map(
            city=city,
            aoi_dict=aoi_dict,
            geo_features=geo_features,
            polygon_features=polygon_features,
            nigeria_path=nigeria_path,
            save_path=city_save_path,
            figsize=(11.69, 7.5),
            dpi=300
        )
        if verbose:
            plt.show()
        plt.close()

    if skipped > 0:
        print(f"\n  \u2713 {skipped} city maps already exist \u2014 skipped.")
    if generated > 0:
        shown = min(generated, VERBOSE_LIMIT)
        print(f"  \u2713 {generated} new maps generated ({shown} shown inline).")
    else:
        print(f"  \u2713 All {len(cities_to_process)} city maps already exist \u2014 nothing to regenerate.")

print(f"\n{'='*80}")
print("STUDY AREA MAP(S) COMPLETE")
print(f"{'='*80}")
if is_test_mode:
    print(f"\u2713 Combined map: {aoi_base_dir / 'Study_Area_Map_Publication.svg'}")
else:
    print(f"\u2713 Overview: {aoi_base_dir / 'Nigeria_Cities_Overview.svg'}")
    print(f"\u2713 Individual maps: City_[Name]/Figures/[Name]_Study_Area_Map.svg")
    print(f"\u2713 Total: 1 overview + {len(cities_to_process)} city maps")


---
## 5. Nigeria Cities & Landscape Overview Map

A national-scale overview with two outputs:

1. **Landscape Overview Map**  Cluster25 landscape classification as background with proportional circles sized by city population (UN WUP 2025 estimates), black point markers at city locations, and city name labels
2. **Cluster25 Classification Table**  Publication-ready table of all 25 landscape clusters with colour swatches, climate class, geology class, and elevation class columns

In [ ]:
# ====================================================================
# NIGERIA CITIES + LANDSCAPE OVERVIEW MAP
# ====================================================================

print(f"\n{'='*80}")
print("CREATING NIGERIA CITIES & LANDSCAPE OVERVIEW MAP")
print(f"{'='*80}\n")

landscape_path = project_root / 'data' / 'inputs' / 'geospatial_features' / 'landscape_ch_polygons_with_class_metrics.geojson'
cities_csv_path = project_root / 'data' / 'inputs' / 'geospatial_features' / 'Major_cities_population_Nigeria_WUP_2025.csv'

# Print first 5 cities
print(f"Cities included ({len(cities_to_process)} total):")
for i, city in enumerate(sorted(cities_to_process)):
    if i < 5:
        print(f"  {i+1}. {city}")
    else:
        break
if len(cities_to_process) > 5:
    print(f"  ... and {len(cities_to_process) - 5} more (see output folder below)")
print()

landscape_save_path = aoi_base_dir / 'Nigeria_Cities_Landscape_Overview'

# --- Landscape overview map ---
landscape_png = landscape_save_path.with_suffix('.svg')
landscape_pdf = landscape_save_path.with_suffix('.svg')

if landscape_png.exists() and landscape_pdf.exists():
    print(f"[SKIP] Landscape overview map already exists:")
    print(f"       {landscape_png.name}")
    print(f"       {landscape_pdf.name}")
else:
    fig_landscape = create_nigeria_cities_landscape_map(
        aoi_dict=aoi_dict,
        cities_csv_path=cities_csv_path,
        landscape_path=landscape_path,
        nigeria_path=nigeria_path,
        cluster25_colors=config.CLUSTER25_COLORS,
        study_year=config.PROJECT_STUDY_YEAR,
        save_path=landscape_save_path,
        title=f'Study Cities on Landscape Classification  Nigeria ({len(cities_to_process)} cities, {config.PROJECT_STUDY_YEAR})',
        figsize=(10, 10),
        dpi=300,
        pop_scale=0.9,
        show_plot=True,
    )
    print(f"[SAVED] {landscape_png.name}")
    print(f"[SAVED] {landscape_pdf.name}")

# --- Cluster25 classification table ---
table_save_path = aoi_base_dir / 'Cluster25_Classification_Table'
table_png = table_save_path.with_suffix('.svg')
table_pdf = table_save_path.with_suffix('.svg')

if table_png.exists() and table_pdf.exists():
    print(f"\n[SKIP] Classification table already exists:")
    print(f"       {table_png.name}")
    print(f"       {table_pdf.name}")
else:
    fig_table = create_cluster25_classification_table(
        landscape_path=landscape_path,
        cluster25_colors=config.CLUSTER25_COLORS,
        save_path=table_save_path,
        title='Cluster25 Landscape Classification',
        figsize=(14, 8),
        dpi=300,
        show_plot=True,
    )
    print(f"\n[SAVED] {table_png.name}")
    print(f"[SAVED] {table_pdf.name}")

print(f"\n{'='*80}")
print("SECTION 5 COMPLETE")
print(f"{'='*80}")

---
## Summary

This notebook completed the **geospatial feature extraction and visualization** workflow for **58 Nigerian city AOIs** (â‰¥300k population, spatially deduplicated).

### Workflow Completed

| Step | Description | Output |
|------|-------------|--------|
| **1. Load AOIs** | Loaded 58 city AOI boundaries from `01_aoi_delineation` | AOI GeoDataFrames with tracking status |
| **2. Feature Extraction** | Downloaded water (JRC/GEE), roads & places (OSM/Overpass) | GeoJSON files per city |
| **3. Individual Maps** | Created feature maps for each city | SVG + SVG per city |
| **4. Publication Map** | Multi-panel study area figure (A4 landscape) | `Study_Area_Map_Publication.svg` |
| **5. Landscape Overview** | National landscape map + Cluster25 classification table | `Nigeria_Cities_Landscape_Overview.svg`, `Cluster25_Classification_Table.svg` |

### Data Sources Used

| Feature | Source | Why This Source? |
|---------|--------|------------------|
| **Water Bodies** | JRC Global Surface Water (GEE) | Consistent satellite-derived extent; captures seasonal water and lagoons reliably |
| **Roads** | OpenStreetMap (Overpass API) | Comprehensive road network; always up-to-date |
| **Places** | OpenStreetMap (Overpass API) | Rich place name database; includes towns, suburbs, villages |
| **Landscape** | Cluster25 Classification (GeoJSON) | 25-class landscape typology with climate, geology, and elevation attributes |

### Output Structure

```
data/outputs/01_aoi_delineation/
â”œâ”€â”€ Study_Area_Map_Publication.svg        # Multi-panel publication figure
â”œâ”€â”€ Nigeria_Cities_Overview.svg           # National overview map
â”œâ”€â”€ Nigeria_Cities_Landscape_Overview.svg # Landscape classification map
â”œâ”€â”€ Cluster25_Classification_Table.svg    # Cluster25 class lookup table
â”œâ”€â”€ Composite_Population_Trend.svg        # Population trend (grouped)
â”œâ”€â”€ Composite_Population_Trend_Individual.svg
â”œâ”€â”€ aoi_manifest.json                         # 58-AOI manifest
â”œâ”€â”€ aoi_features_tracking.json                # Skip logic tracking
â””â”€â”€ City_[Name]/
    â”œâ”€â”€ Features/
    â”‚   â”œâ”€â”€ [Name]_AOI_Boundary.geojson
    â”‚   â”œâ”€â”€ [Name]_Main_Polygon.geojson
    â”‚   â”œâ”€â”€ [Name]_Adjacent_Polygons.geojson
    â”‚   â””â”€â”€ OSM/
    â”‚       â”œâ”€â”€ [Name]_water_bodies.geojson   â† JRC via GEE
    â”‚       â”œâ”€â”€ [Name]_roads.geojson          â† OSM via Overpass
    â”‚       â””â”€â”€ [Name]_places.geojson         â† OSM via Overpass
    â””â”€â”€ Figures/
        â””â”€â”€ [Name]_geospatial_features_map.svg
```

### Key Features

- **Skip Logic**: Features only re-extracted when AOI area changes (tracked via JSON); output figures skipped if SVG + SVG exist
- **Fast Queries**: Overpass API completes in 10â€“30s per city
- **Publication-Ready**: A4 landscape figure with scale bars inside map frames
- **Landscape Context**: National-scale Cluster25 classification map showing all 58 study cities in their landscape setting